# Problem 1. 베이징 초미세먼지(PM2.5) 농도 예측하기

## 문제 개요
환경 연구팀은 기상 관측값만으로 **시간별 초미세먼지(PM2.5) 농도**를 추정하는 모델을 만들려 합니다.
2010~2013년 베이징의 시간별 기상·PM2.5 기록으로 학습하고, **2014년 1년간의 PM2.5 농도**를 예측하세요.

## 파일 구조
```
Problem1/
├── problem1.ipynb      ← 이 노트북
├── submission.csv        ← 제출 파일 (직접 생성)
└── data/
    ├── train.csv
    ├── test.csv
    └── sample_submission.csv
```

## 데이터
- `data/train.csv` : 2010-01-01 ~ 2013-12-31, 35,064시간 (pm25 포함, 일부 결측)
- `data/test.csv` : 2014-01-01 ~ 2014-12-31, 8,760시간 (pm25 없음)

| 컬럼 | 설명 |
|---|---|
| id | 시간 순서 번호 (연속, 1시간 간격) |
| year, month, day, hour | 관측 시각 |
| **pm25** | **PM2.5 농도(㎍/㎥), 예측 대상** |
| DEWP | 이슬점(℃) |
| TEMP | 기온(℃) |
| PRES | 기압(hPa) |
| cbwd | 풍향 (NE, NW, SE, cv=무풍·불규칙) |
| Iws | 같은 풍향이 지속된 동안의 누적 풍속(m/s) |
| Is / Ir | 눈·비가 이어진 누적 시간 |

### 데이터 특징
- 테스트 기간의 **기상 자료는 1시간도 빠짐없이** 주어집니다. PM2.5만 없습니다.
- 학습 데이터의 pm25에는 결측이 약 5% 있습니다.
- 미세먼지는 한 시간의 날씨보다 **며칠간 쌓인 기상 조건**에 크게 좌우됩니다.
- 출처: Beijing PM2.5 Data (UCI Machine Learning Repository, CC BY 4.0)

## 규칙
- 제한 시간: 3문제 합계 **90분**
- 허용: pandas, scikit-learn, LightGBM, XGBoost, OpenCV, PyTorch 등 일반 라이브러리와 공개 사전학습 모델
- 금지: LLM API로 테스트 데이터를 직접 예측, 테스트 데이터 수작업 라벨링, 원본 공개 데이터셋에서 정답 조회
- 결과는 이 노트북과 같은 위치(`Problem1/`)에 **`submission.csv`** 로 저장

## 평가
**MAE (평균 절대 오차)** — 낮을수록 좋습니다.

- 2014년 중 실측값이 없는 99시간은 채점에서 제외합니다. 제출 파일에는 모든 id를 포함하세요.

## 초기 코드 (베이스라인)
아래 베이스라인은 그 시각의 기상값 6개만으로 선형회귀를 합니다. 동작은 하지만 점수가 낮습니다.

In [ ]:
import pandas as pd
from sklearn.linear_model import LinearRegression
DATA_DIR = "data"
train = pd.read_csv(f"{DATA_DIR}/train.csv").dropna(subset=["pm25"]); test = pd.read_csv(f"{DATA_DIR}/test.csv")
num = ["DEWP", "TEMP", "PRES", "Iws", "Is", "Ir"]          # 그 시각의 기상값만 사용
model = LinearRegression().fit(train[num], train.pm25)
pd.DataFrame({"id": test.id, "pm25": model.predict(test[num])}).to_csv("submission.csv", index=False)

## 제출 형식
`submission.csv`

| id | pm25 |
|---|---|
| 35064 | 120.5 |
| 35065 | 98.0 |

- 컬럼명 `id`, `pm25`, 결측 없이 모든 id 포함
- 포맷이 틀리면 0점 처리됩니다.

In [ ]:
import pandas as pd
sub = pd.read_csv("submission.csv")
print(sub.shape)
sub.head()